# Incident Response Runbook: Impact - Denial of Service

**Tactic:** Impact
**Technique:** Denial of Service (T1499)
**Severity:** CRITICAL

## Overview

This runbook provides step-by-step incident response procedures for Denial of Service activities.

## Incident Response Phases

1. **Detection & Analysis**
2. **Containment**
3. **Eradication**
4. **Recovery**
5. **Post-Incident Activities**


## Phase 1: Detection & Analysis

### Objectives
- Confirm the incident
- Identify the scope
- Assess impact


In [ ]:
import json
import re
from datetime import datetime, timedelta
import sys
import os

# Add parent directory to path for imports
sys.path.append(os.path.dirname(os.path.dirname(os.path.abspath(__file__))))

from splunk.splunk_data_collector import SplunkDataCollector
from crowdstrike.crowdstrike_data_collector import CrowdStrikeDataCollector
from misp.misp_integration import MISPIntegration
from iris.iris_integration import IRISIntegration
from shuffle.shuffle_integration import ShuffleIntegration

# Step 1: Initial Alert Analysis & Detection
print("=" * 60)
print("STEP 1: Initial Alert Analysis & Detection")
print("=" * 60)

# Initialize security integrations
splunk = SplunkDataCollector()
crowdstrike = CrowdStrikeDataCollector()
misp = MISPIntegration()
iris = IRISIntegration()
shuffle = ShuffleIntegration()

# Create incident case in IRIS
incident_data = {
    'title': 'Denial of Service Attack - T1499',
    'description': 'Potential DDoS attack detected - high volume traffic and service degradation identified',
    'severity': 'HIGH',
    'category': 'Network Attack',
    'tags': ['ddos', 'denial-of-service', 'network-attack', 't1499']
}

case_id = iris.create_case(incident_data)
print(f" Created IRIS case: {case_id}")

# Detection queries for DDoS indicators
print("\n Executing DDoS detection queries...")

# Splunk query for traffic volume anomalies
traffic_anomaly_query = """
index=* sourcetype=network_traffic
| timechart span=1m count by dest_ip
| where count > 10000
| stats avg(count) as avg_traffic, max(count) as peak_traffic, count as spike_count
"""

traffic_results = splunk.execute_query(traffic_anomaly_query)
print(f" Found {len(traffic_results)} traffic anomalies")

# Splunk query for SYN flood detection
syn_flood_query = """
index=* sourcetype=network_traffic
| where protocol="TCP" AND tcp_flags="SYN"
| stats count by src_ip, dest_ip, dest_port
| where count > 1000
| sort -count
"""

syn_results = splunk.execute_query(syn_flood_query)
print(f" Found {len(syn_results)} potential SYN flood sources")

# Splunk query for UDP flood detection
udp_flood_query = """
index=* sourcetype=network_traffic
| where protocol="UDP"
| stats count by src_ip, dest_port
| where count > 5000
| sort -count
"""

udp_results = splunk.execute_query(udp_flood_query)
print(f" Found {len(udp_results)} potential UDP flood sources")

# Splunk query for HTTP flood detection
http_flood_query = """
index=* sourcetype=web_access
| where status=200
| timechart span=1m count
| where count > 1000
"""

http_results = splunk.execute_query(http_flood_query)
print(f" Found {len(http_results)} HTTP flood patterns")

# Check for service availability issues
service_query = """
index=* sourcetype=service_monitoring
| where status="down" OR response_time > 5000
| stats count by service_name, host
| sort -count
"""

service_results = splunk.execute_query(service_query)
print(f" Found {len(service_results)} service availability issues")

# Check for known DDoS IOCs in MISP
ddos_indicators = misp.search_indicators("ddos", limit=50)
print(f" Retrieved {len(ddos_indicators)} DDoS indicators from MISP")

# Compile detection results
detection_summary = {
    'case_id': case_id,
    'timestamp': datetime.now().isoformat(),
    'tactic': 'Impact',
    'technique': 'Denial of Service (T1499)',
    'severity': 'HIGH',
    'indicators': {
        'traffic_anomalies': len(traffic_results),
        'syn_flood_sources': len(syn_results),
        'udp_flood_sources': len(udp_results),
        'http_flood_patterns': len(http_results),
        'service_issues': len(service_results),
        'misp_indicators': len(ddos_indicators)
    },
    'affected_services': list(set([r.get('service_name', 'unknown') for r in service_results])),
    'attack_vector': 'DDoS' if len(traffic_results) > 0 else 'DoS'
}

print(f"\n📊 Detection Summary:")
print(f"  Case ID: {detection_summary['case_id']}")
print(f"  Severity: {detection_summary['severity']}")
print(f"  Attack Vector: {detection_summary['attack_vector']}")
print(f"  Traffic Anomalies: {detection_summary['indicators']['traffic_anomalies']}")
print(f"  SYN Flood Sources: {detection_summary['indicators']['syn_flood_sources']}")
print(f"  UDP Flood Sources: {detection_summary['indicators']['udp_flood_sources']}")
print(f"  Service Issues: {detection_summary['indicators']['service_issues']}")

# Trigger Shuffle workflow for automated DDoS response
if detection_summary['indicators']['traffic_anomalies'] > 0:
    shuffle.trigger_workflow('ddos_detection_response', detection_summary)
    print(" Triggered automated DDoS response workflow")


## Phase 2: Containment

### Objectives
- Stop the attack
- Isolate affected systems
- Prevent spread


In [ ]:
# Step 2: Containment Actions
print("\n" + "=" * 60)
print("STEP 2: Containment Actions")
print("=" * 60)

containment_actions = []

# Activate DDoS mitigation via Shuffle
print("\n Activating DDoS mitigation...")
try:
    mitigation_config = {
        'attack_type': detection_summary['attack_vector'],
        'traffic_threshold': 10000,
        'rate_limiting': True,
        'ip_blocking': True,
        'auto_scaling': True
    }
    mitigation_result = shuffle.activate_ddos_mitigation(mitigation_config)
    if mitigation_result.get('success'):
        containment_actions.append(" DDoS mitigation activated")
        print("   DDoS mitigation activated")
    else:
        containment_actions.append(" DDoS mitigation activation failed")
        print("   DDoS mitigation activation failed")
except Exception as e:
    containment_actions.append(f" Error activating mitigation: {str(e)}")
    print(f"   Error activating mitigation: {str(e)}")

# Block malicious IPs identified in SYN floods
print("\n Blocking SYN flood sources...")
for result in syn_results[:50]:  # Limit to top 50 to avoid overload
    try:
        block_result = shuffle.block_ip(result['src_ip'])
        if block_result.get('success'):
            containment_actions.append(f" Blocked SYN flood IP: {result['src_ip']}")
            print(f"   Blocked SYN flood IP: {result['src_ip']}")
    except Exception as e:
        containment_actions.append(f" Failed to block IP {result['src_ip']}: {str(e)}")
        print(f"   Failed to block IP {result['src_ip']}: {str(e)}")

# Block UDP flood sources
print("\n Blocking UDP flood sources...")
for result in udp_results[:50]:  # Limit to top 50
    try:
        block_result = shuffle.block_ip(result['src_ip'])
        if block_result.get('success'):
            containment_actions.append(f" Blocked UDP flood IP: {result['src_ip']}")
            print(f"   Blocked UDP flood IP: {result['src_ip']}")
    except Exception as e:
        containment_actions.append(f" Failed to block IP {result['src_ip']}: {str(e)}")
        print(f"   Failed to block IP {result['src_ip']}: {str(e)}")

# Implement rate limiting
print("\n⏱ Implementing rate limiting...")
try:
    rate_limit_config = {
        'http_requests_per_minute': 1000,
        'tcp_connections_per_minute': 5000,
        'udp_packets_per_minute': 10000,
        'duration_minutes': 60
    }
    rate_limit_result = shuffle.implement_rate_limiting(rate_limit_config)
    if rate_limit_result.get('success'):
        containment_actions.append(" Rate limiting implemented")
        print("   Rate limiting implemented")
except Exception as e:
    containment_actions.append(f" Failed to implement rate limiting: {str(e)}")
    print(f"   Failed to implement rate limiting: {str(e)}")

# Enable traffic scrubbing
print("\n Enabling traffic scrubbing...")
try:
    scrubbing_config = {
        'scrub_invalid_packets': True,
        'filter_known_attack_patterns': True,
        'anomaly_detection': True
    }
    scrubbing_result = shuffle.enable_traffic_scrubbing(scrubbing_config)
    if scrubbing_result.get('success'):
        containment_actions.append(" Traffic scrubbing enabled")
        print("   Traffic scrubbing enabled")
except Exception as e:
    containment_actions.append(f" Failed to enable traffic scrubbing: {str(e)}")
    print(f"   Failed to enable traffic scrubbing: {str(e)}")

# Scale infrastructure if needed
print("\n📈 Scaling infrastructure...")
try:
    scale_config = {
        'auto_scale': True,
        'max_instances': 10,
        'scale_up_threshold': 80,  # CPU utilization
        'scale_down_threshold': 30
    }
    scale_result = shuffle.scale_infrastructure(scale_config)
    if scale_result.get('success'):
        containment_actions.append(" Infrastructure scaling activated")
        print("   Infrastructure scaling activated")
except Exception as e:
    containment_actions.append(f" Failed to scale infrastructure: {str(e)}")
    print(f"   Failed to scale infrastructure: {str(e)}")

# Update IRIS case with containment actions
iris.update_case(case_id, {
    'containment_actions': containment_actions,
    'containment_timestamp': datetime.now().isoformat(),
    'status': 'containment_in_progress'
})

print(f"\n Containment Summary:")
print(f"  Actions Taken: {len([a for a in containment_actions if a.startswith('')])}")
print(f"  Warnings: {len([a for a in containment_actions if a.startswith('')])}")
print(f"  Errors: {len([a for a in containment_actions if a.startswith('')])}")

# Trigger Shuffle workflow for containment verification
shuffle.trigger_workflow('ddos_containment_verification', {
    'case_id': case_id,
    'actions_taken': containment_actions
})
print(" Triggered containment verification workflow")


## Phase 3: Eradication

### Objectives
- Remove malicious artifacts
- Close vulnerabilities
- Verify threat removal


In [ ]:
# Step 3: Eradication Actions
print("\n" + "=" * 60)
print("STEP 3: Eradication Actions")
print("=" * 60)

eradication_actions = []

# Identify and block botnet C2 servers
print("\n Identifying and blocking botnet C2 servers...")
c2_query = """
index=* sourcetype=network_traffic
| where dest_port IN (6667, 6666, 6665, 6669, 7000, 8000, 8080)
| stats count by dest_ip, dest_port
| where count > 100
| sort -count
"""

c2_results = splunk.execute_query(c2_query)
for c2_server in c2_results:
    try:
        block_result = shuffle.block_ip(c2_server['dest_ip'])
        if block_result.get('success'):
            eradication_actions.append(f" Blocked C2 server: {c2_server['dest_ip']}:{c2_server['dest_port']}")
            print(f"   Blocked C2 server: {c2_server['dest_ip']}:{c2_server['dest_port']}")
    except Exception as e:
        eradication_actions.append(f" Failed to block C2 server {c2_server['dest_ip']}: {str(e)}")
        print(f"   Failed to block C2 server {c2_server['dest_ip']}: {str(e)}")

# Clean up malicious traffic patterns
print("\n Cleaning up malicious traffic patterns...")
try:
    cleanup_config = {
        'flush_connection_queues': True,
        'clear_attack_queues': True,
        'reset_rate_limits': False  # Keep rate limits active
    }
    cleanup_result = shuffle.cleanup_malicious_traffic(cleanup_config)
    if cleanup_result.get('success'):
        eradication_actions.append(" Malicious traffic patterns cleaned")
        print("   Malicious traffic patterns cleaned")
except Exception as e:
    eradication_actions.append(f" Failed to clean traffic patterns: {str(e)}")
    print(f"   Failed to clean traffic patterns: {str(e)}")

# Update firewall rules
print("\n Updating firewall rules...")
try:
    firewall_rules = {
        'block_ddos_sources': True,
        'enable_geo_filtering': True,
        'strict_rate_limiting': True,
        'block_suspicious_ports': [19, 53, 123, 1900]  # Common DDoS ports
    }
    firewall_result = shuffle.update_firewall_rules(firewall_rules)
    if firewall_result.get('success'):
        eradication_actions.append(" Firewall rules updated")
        print("   Firewall rules updated")
except Exception as e:
    eradication_actions.append(f" Failed to update firewall rules: {str(e)}")
    print(f"   Failed to update firewall rules: {str(e)}")

# Implement traffic shaping
print("\n📊 Implementing traffic shaping...")
try:
    shaping_config = {
        'prioritize_legitimate_traffic': True,
        'throttle_suspicious_sources': True,
        'bandwidth_allocation': {
            'http': 70,
            'api': 20,
            'other': 10
        }
    }
    shaping_result = shuffle.implement_traffic_shaping(shaping_config)
    if shaping_result.get('success'):
        eradication_actions.append(" Traffic shaping implemented")
        print("   Traffic shaping implemented")
except Exception as e:
    eradication_actions.append(f" Failed to implement traffic shaping: {str(e)}")
    print(f"   Failed to implement traffic shaping: {str(e)}")

# Verify attack cessation
print("\n✅ Verifying attack cessation...")
verification_results = []
current_traffic = splunk.execute_query(traffic_anomaly_query)
if len(current_traffic) == 0:
    verification_results.append(" DDoS attack traffic reduced to normal levels")
    print("   DDoS attack traffic reduced to normal levels")
else:
    verification_results.append(f" Attack traffic still elevated: {len(current_traffic)} anomalies")
    print(f"   Attack traffic still elevated: {len(current_traffic)} anomalies")

# Check service availability
service_status = splunk.execute_query(service_query)
if len(service_status) == 0:
    verification_results.append(" All services restored to normal operation")
    print("   All services restored to normal operation")
else:
    verification_results.append(f" Services still experiencing issues: {len(service_status)}")
    print(f"   Services still experiencing issues: {len(service_status)}")

# Update IRIS case with eradication actions
iris.update_case(case_id, {
    'eradication_actions': eradication_actions,
    'verification_results': verification_results,
    'eradication_timestamp': datetime.now().isoformat(),
    'status': 'eradication_complete'
})

print(f"\n Eradication Summary:")
print(f"  Actions Taken: {len([a for a in eradication_actions if a.startswith('')])}")
print(f"  Warnings: {len([a for a in eradication_actions if a.startswith('')])}")
print(f"  Errors: {len([a for a in eradication_actions if a.startswith('')])}")
print(f"  Verification Results: {len([r for r in verification_results if r.startswith('')])} passed")

# Share indicators with MISP
if len(ddos_indicators) > 0:
    misp.share_indicators(ddos_indicators, case_id)
    print(" Shared DDoS indicators with MISP community")


## Phase 4: Recovery

### Objectives
- Restore systems
- Validate functionality
- Monitor for issues


In [ ]:
# Step 4: Recovery Actions
print("\n" + "=" * 60)
print("STEP 4: Recovery Actions")
print("=" * 60)

recovery_actions = []

# Gradually reduce mitigation measures
print("\n Gradually reducing mitigation measures...")
try:
    reduction_config = {
        'gradual_rollback': True,
        'monitor_traffic': True,
        'rollback_duration_minutes': 30,
        'alert_threshold': 5000  # Lower threshold for monitoring
    }
    reduction_result = shuffle.gradual_mitigation_reduction(reduction_config)
    if reduction_result.get('success'):
        recovery_actions.append(" Gradual mitigation reduction initiated")
        print("   Gradual mitigation reduction initiated")
    else:
        recovery_actions.append(" Gradual mitigation reduction failed")
        print("   Gradual mitigation reduction failed")
except Exception as e:
    recovery_actions.append(f" Error reducing mitigation: {str(e)}")
    print(f"   Error reducing mitigation: {str(e)}")

# Restore normal traffic flow
print("\n🌊 Restoring normal traffic flow...")
try:
    restore_config = {
        'remove_rate_limits': False,  # Keep some rate limiting
        'restore_bandwidth_allocation': True,
        'disable_traffic_shaping': False,  # Keep shaping for monitoring
        'normal_operation_mode': True
    }
    restore_result = shuffle.restore_normal_traffic_flow(restore_config)
    if restore_result.get('success'):
        recovery_actions.append(" Normal traffic flow restored")
        print("   Normal traffic flow restored")
except Exception as e:
    recovery_actions.append(f" Failed to restore traffic flow: {str(e)}")
    print(f"   Failed to restore traffic flow: {str(e)}")

# Validate service performance
print("\n Validating service performance...")
performance_checks = []
for service in detection_summary['affected_services']:
    try:
        perf_result = splunk.check_service_performance(service)
        if perf_result.get('performance_normal'):
            performance_checks.append(f" Service {service} performance normal")
            print(f"   Service {service} performance normal")
        else:
            performance_checks.append(f" Service {service} performance issues: {perf_result.get('issues', [])}")
            print(f"   Service {service} performance issues: {perf_result.get('issues', [])}")
    except Exception as e:
        performance_checks.append(f" Performance check failed for {service}: {str(e)}")
        print(f"   Performance check failed for {service}: {str(e)}")

# Monitor for attack recurrence
print("\n Establishing monitoring for attack recurrence...")
try:
    recurrence_monitoring = {
        'traffic_patterns': True,
        'service_performance': True,
        'duration_hours': 72,
        'alert_sensitivity': 'medium',
        'indicators': ['traffic_spikes', 'service_degradation', 'ddos_patterns']
    }
    splunk.setup_recurrence_monitoring(recurrence_monitoring)
    recovery_actions.append(" Recurrence monitoring established")
    print("   Recurrence monitoring established")
except Exception as e:
    recovery_actions.append(f" Failed to establish recurrence monitoring: {str(e)}")
    print(f"   Failed to establish recurrence monitoring: {str(e)}")

# Implement long-term DDoS protection
print("\n Implementing long-term DDoS protection...")
try:
    protection_config = {
        'always_on_protection': True,
        'adaptive_learning': True,
        'cloud_mitigation': True,
        'regular_testing': True
    }
    protection_result = shuffle.implement_ddos_protection(protection_config)
    if protection_result.get('success'):
        recovery_actions.append(" Long-term DDoS protection implemented")
        print("   Long-term DDoS protection implemented")
except Exception as e:
    recovery_actions.append(f" Failed to implement long-term protection: {str(e)}")
    print(f"   Failed to implement long-term protection: {str(e)}")

# Conduct traffic analysis
print("\n📈 Conducting traffic analysis...")
try:
    analysis_config = {
        'analyze_attack_patterns': True,
        'identify_attack_sources': True,
        'generate_traffic_report': True,
        'time_range_hours': 24
    }
    analysis_result = splunk.conduct_traffic_analysis(analysis_config)
    if analysis_result.get('success'):
        recovery_actions.append(" Traffic analysis completed")
        print("   Traffic analysis completed")
except Exception as e:
    recovery_actions.append(f" Traffic analysis failed: {str(e)}")
    print(f"   Traffic analysis failed: {str(e)}")

# Update IRIS case with recovery actions
iris.update_case(case_id, {
    'recovery_actions': recovery_actions,
    'performance_checks': performance_checks,
    'recovery_timestamp': datetime.now().isoformat(),
    'status': 'recovery_complete'
})

print(f"\n Recovery Summary:")
print(f"  Actions Taken: {len([a for a in recovery_actions if a.startswith('')])}")
print(f"  Warnings: {len([a for a in recovery_actions if a.startswith('')])}")
print(f"  Errors: {len([a for a in recovery_actions if a.startswith('')])}")
print(f"  Performance Checks Passed: {len([c for c in performance_checks if c.startswith('')])}")

# Trigger Shuffle workflow for recovery verification
shuffle.trigger_workflow('ddos_recovery_verification', {
    'case_id': case_id,
    'actions_taken': recovery_actions,
    'performance_checks': performance_checks
})
print(" Triggered recovery verification workflow")


## Phase 5: Post-Incident Activities

### Objectives
- Document findings
- Implement improvements
- Share lessons learned


In [ ]:
# Step 5: Post-Incident Actions
print("\n" + "=" * 60)
print("STEP 5: Post-Incident Actions")
print("=" * 60)

post_incident_actions = []

# Generate incident report
print("\n Generating incident report...")
try:
    incident_report = {
        'case_id': case_id,
        'title': 'DDoS Attack Incident Response Report',
        'timeline': {
            'detection': detection_summary['timestamp'],
            'containment': datetime.now().isoformat(),  # Would be actual timestamp
            'eradication': datetime.now().isoformat(),   # Would be actual timestamp
            'recovery': datetime.now().isoformat(),      # Would be actual timestamp
            'closure': datetime.now().isoformat()
        },
        'impact_assessment': {
            'attack_duration': 'TBD',  # Would calculate actual duration
            'peak_traffic': max([r.get('peak_traffic', 0) for r in traffic_results]) if traffic_results else 0,
            'services_affected': len(detection_summary['affected_services']),
            'business_impact': 'MEDIUM',  # Would be determined by business impact analysis
            'data_loss': False,
            'service_downtime': 'TBD'  # Would calculate actual downtime
        },
        'response_metrics': {
            'detection_to_containment': 'TBD',  # Would calculate actual time
            'containment_to_eradication': 'TBD',
            'total_resolution_time': 'TBD',
            'automation_level': 'HIGH'
        },
        'attack_characteristics': {
            'attack_vector': detection_summary['attack_vector'],
            'traffic_volume': detection_summary['indicators']['traffic_anomalies'],
            'source_ips': detection_summary['indicators']['syn_flood_sources'] + detection_summary['indicators']['udp_flood_sources'],
            'targeted_ports': 'Multiple',
            'mitigation_effectiveness': 'HIGH'
        },
        'lessons_learned': [
            'Automated DDoS mitigation activated within minutes',
            'Rate limiting and traffic scrubbing were effective',
            'Infrastructure auto-scaling prevented complete outage',
            'MISP integration provided valuable attack intelligence'
        ]
    }

    report_id = iris.generate_report(case_id, incident_report)
    post_incident_actions.append(f" Generated incident report: {report_id}")
    print(f"   Generated incident report: {report_id}")

except Exception as e:
    post_incident_actions.append(f" Failed to generate report: {str(e)}")
    print(f"   Failed to generate report: {str(e)}")

# Document lessons learned
print("\n Documenting lessons learned...")
lessons_learned = [
    "Implement always-on DDoS protection",
    "Regular DDoS simulation testing",
    "Cloud-based mitigation for scalability",
    "Automated traffic analysis and reporting",
    "Integration with threat intelligence for early warning",
    "Regular review of rate limiting policies"
]

try:
    iris.add_lessons_learned(case_id, lessons_learned)
    post_incident_actions.append(" Documented lessons learned")
    print("   Documented lessons learned")
except Exception as e:
    post_incident_actions.append(f" Failed to document lessons learned: {str(e)}")
    print(f"   Failed to document lessons learned: {str(e)}")

# Implement preventive measures
print("\n Implementing preventive measures...")
preventive_measures = []

try:
    # Update DDoS protection policies
    policy_updates = {
        'ddos_protection_level': 'maximum',
        'traffic_monitoring': 'continuous',
        'automated_response': True,
        'regular_penetration_testing': True
    }
    shuffle.update_security_policies(policy_updates)
    preventive_measures.append(" Updated DDoS protection policies")
    print("   Updated DDoS protection policies")

    # Enhance monitoring rules
    monitoring_updates = {
        'ddos_detection_sensitivity': 'high',
        'traffic_anomaly_detection': True,
        'automated_mitigation': True
    }
    splunk.update_monitoring_rules(monitoring_updates)
    preventive_measures.append(" Enhanced monitoring rules")
    print("   Enhanced monitoring rules")

    # Update threat intelligence feeds
    misp.update_feeds(['ddos', 'botnet', 'amplification_attacks'])
    preventive_measures.append(" Updated threat intelligence feeds")
    print("   Updated threat intelligence feeds")

except Exception as e:
    preventive_measures.append(f" Failed to implement preventive measures: {str(e)}")
    print(f"   Failed to implement preventive measures: {str(e)}")

# Conduct post-incident review
print("\n Conducting post-incident review...")
try:
    review_findings = {
        'effectiveness_rating': 'HIGH',
        'areas_for_improvement': [
            'Earlier attack detection',
            'Better traffic analysis automation',
            'Improved reporting granularity'
        ],
        'recommendations': [
            'Implement AI-based traffic anomaly detection',
            'Regular DDoS drills and simulations',
            'Enhanced integration with cloud providers',
            'Automated compliance and regulatory reporting'
        ]
    }

    iris.conduct_post_incident_review(case_id, review_findings)
    post_incident_actions.append(" Conducted post-incident review")
    print("   Conducted post-incident review")

except Exception as e:
    post_incident_actions.append(f" Failed to conduct review: {str(e)}")
    print(f"   Failed to conduct review: {str(e)}")

# Close the incident case
print("\n Closing incident case...")
try:
    closure_data = {
        'closure_reason': 'DDoS attack mitigated - traffic normalized, services restored',
        'closure_timestamp': datetime.now().isoformat(),
        'final_status': 'CLOSED',
        'follow_up_required': False,
        'reopen_criteria': 'Detection of similar DDoS attack patterns'
    }

    iris.close_case(case_id, closure_data)
    post_incident_actions.append(" Closed incident case")
    print("   Closed incident case")

except Exception as e:
    post_incident_actions.append(f" Failed to close case: {str(e)}")
    print(f"   Failed to close case: {str(e)}")

# Final summary
print(f"\n Post-Incident Summary:")
print(f"  Case ID: {case_id}")
print(f"  Actions Completed: {len([a for a in post_incident_actions if a.startswith('')])}")
print(f"  Warnings: {len([a for a in post_incident_actions if a.startswith('')])}")
print(f"  Errors: {len([a for a in post_incident_actions if a.startswith('')])}")
print(f"  Preventive Measures: {len(preventive_measures)}")
print(f"  Lessons Learned: {len(lessons_learned)}")

print("\n Incident Response Complete")
print("=" * 60)
print("DDoS attack successfully mitigated and resolved.")
print("Traffic normalized, services restored, and preventive measures implemented.")
print("=" * 60)


## Summary

This incident response runbook has guided you through the complete lifecycle of responding to this incident.

### Key Takeaways
- Rapid detection and response are critical
- Containment prevents further damage
- Thorough eradication prevents reinfection
- Continuous monitoring ensures recovery

### Next Steps
- Review and approve the incident report
- Implement recommended preventive measures
- Schedule follow-up review
- Share lessons learned with the team
